In [1]:
import pyspark 
import pandas as pd
from pyspark.sql import SparkSession,Row,DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *


spark=SparkSession.builder.master("local[1]")\
        .appName("Netflix")\
        .config("spark.sql.warehouse.dir", "/home/jovyan/work/database") \
        .config("spark.jars", "/Users/eduardoalberto/opt/spark-4.0.0/jars/postgresql-42.7.3.jar")\
        .enableHiveSupport()\
        .getOrCreate()
sc = spark.sparkContext
spark.sparkContext.setLogLevel("OFF") 
print('PySpark Version :'+spark.version)
print('PySpark Version :'+spark.sparkContext.version)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/31 22:40:41 WARN Utils: Your hostname, MacBook-Pro-de-Eduardo.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.108 instead (on interface en8)
25/08/31 22:40:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/08/31 22:40:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


PySpark Version :4.0.0
PySpark Version :4.0.0


In [ ]:
spark.read.csv('/Users/eduardoalberto/LoadFile/input/netflix_titles_clean.csv',header=True,inferSchema=True,sep="," ,quote= '"',escape='"')\
            .createOrReplaceTempView("tb_netflix_titles")

In [ ]:
# !pip install --upgrade "pandas>=2.0.0"

#!pip install --upgrade pyspark==4.0.0
!pip3.9 list


In [ ]:
# spark.table("tb_netflix_titles").filter(F.col("date_added") == "Toni Tones").show()


df = (
    spark.table("tb_netflix_titles")
        .withColumn("id", F.substring(F.col("show_id"), 2, 4))
        .withColumn("date_added", F.trim(F.col("date_added")))
        # só converte valores que correspondem ao padrão de data
        .withColumn(
            "dt_added",
            F.when(
                F.col("date_added").rlike("^[A-Za-z]+ [0-9]{1,2}, [0-9]{4}$"),
                F.to_date(F.col("date_added"), "MMMM d, yyyy")
            ).otherwise(None)
        )
)




df.write.mode("overwrite").parquet("/Users/eduardoalberto/LoadFile/repository/teste/")
df.toPandas()




In [ ]:
spark.table("tb_netflix_titles").select("rating").distinct().toPandas()

In [ ]:
! ls /Users/eduardoalberto/LoadFile/output/netflix/processados/arq/data_execucao=2025-08-25



In [2]:
df = spark.read.parquet("/Users/eduardoalberto/LoadFile/output/netflix/processados/arq/data_execucao=2025-08-25/*.parquet")

df.toPandas()



,id,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,dt_added,dt_processamento,country_name,total_shows
0,8,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",2021-09-24,2025-08-25 21:10:53.778382,"United States, Ghana, Burkina Faso, United Kin...",1
1,9,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...,2021-09-24,2025-08-25 21:10:53.778382,United Kingdom,1
2,10,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris ODowd, Kevin Kline, Ti...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...,2021-09-24,2025-08-25 21:10:53.778382,United States,1
3,13,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic","September 23, 2021",2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...,2021-09-23,2025-08-25 21:10:53.778382,"Germany, Czech Republic",1
4,25,s25,Movie,Jeans,S. Shankar,"Prashanth, Aishwarya Rai Bachchan, Sri Lakshmi...",India,"September 21, 2021",1998,TV-14,166 min,"Comedies, International Movies, Romantic Movies",When the father of the man she loves insists t...,2021-09-21,2025-08-25 21:10:53.778382,India,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5332,8802,s8802,Movie,Zinzana,Majid Al Ansari,"Ali Suliman, Saleh Bakri, Yasa, Ali Al-Jabri, ...","United Arab Emirates, Jordan","March 9, 2016",2015,TV-MA,96 min,"Dramas, International Movies, Thrillers",Recovering alcoholic Talal wakes up inside a s...,2016-03-09,2025-08-25 21:10:53.778382,"United Arab Emirates, Jordan",1
5333,8803,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...",2019-11-20,2025-08-25 21:10:53.778382,United States,1
5334,8805,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,2019-11-01,2025-08-25 21:10:53.778382,United States,1
5335,8806,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...",2020-01-11,2025-08-25 21:10:53.778382,United States,1


### valida DQ

In [4]:
import os
props = {
    "user": os.getenv("DB_USER", "dbpostgres"),
    "password": os.getenv("DB_PASSWORD", "postgre123"),
    "driver": "org.postgresql.Driver"
}

spark.read.jdbc("jdbc:postgresql://localhost:5432/dbpostgres", "data_quality_report",properties=props).createOrReplaceTempView("tb_dq_report")
spark.table("tb_dq_report").toPandas()


,coluna,valor,count,quality_flag
0,rating,TV-Y,76,OK
1,rating,UR,3,OK
2,rating,"Classic Movies, Documentaries",1,OK
3,rating,None,1,OK
4,rating,PG,275,OK
...,...,...,...,...
35230,listed_in,"Anime Series, International TV Shows",5,OK
35231,listed_in,"Comedies, Sci-Fi & Fantasy",3,OK
35232,listed_in,"Romantic TV Shows, TV Comedies, TV Dramas",1,OK
35233,listed_in,"Dramas, International Movies",336,OK
